# Visualise the recorded garage dataset

Replays one episode in the simulation (joint actions from the dataset). The overlaid camera images (top-right agent view,
bottom-right wrist view) come from the dataset, so you can compare them with the re-simulated scene.

In [13]:
import numpy as np, torch
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from lerobot.common.datasets.utils import write_json, serialize_dict
from mujoco_env.y_env_tools import SimpleEnvTools

ROOT = './demo_data_garage'
dataset = LeRobotDataset('garage_pnp', root=ROOT)
print("episodes:", dataset.meta.total_episodes, " frames:", dataset.num_frames)
print("tasks:", dataset.meta.tasks)

Resolving data files:   0%|          | 0/200 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset

In [9]:
class EpisodeSampler(torch.utils.data.Sampler):
    def __init__(self, dataset, episode_index):
        self.frame_ids = range(dataset.episode_data_index["from"][episode_index].item(),
                               dataset.episode_data_index["to"][episode_index].item())
    def __iter__(self): return iter(self.frame_ids)
    def __len__(self): return len(self.frame_ids)

In [10]:
episode_index = 6            # <- change me, re-run this cell and the replay cell

episode_sampler = EpisodeSampler(dataset, episode_index)
dataloader = torch.utils.data.DataLoader(dataset, num_workers=1, batch_size=1, sampler=episode_sampler)

In [11]:
PnPEnv = SimpleEnvTools('./asset/example_scene_tools.xml', action_type='joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[39] n_qvel:[35] n_qacc:[35] n_ctrl:[11]
 integrator:[IMPLICITFAST]

n_body:[24]
 [0/24] [world] mass:[0.00]kg
 [1/24] [front_object_table] mass:[1.00]kg
 [2/24] [camera] mass:[0.00]kg
 [3/24] [camera2] mass:[0.00]kg
 [4/24] [camera3] mass:[0.00]kg
 [5/24] [slider_carriage] mass:[12.00]kg
 [6/24] [link1] mass:[2.06]kg
 [7/24] [link2] mass:[3.68]kg
 [8/24] [link3] mass:[2.39]kg
 [9/24] [link4] mass:[1.40]kg
 [10/24] [link5] mass:[1.40]kg
 [11/24] [link6] mass:[0.65]kg
 [12/24] [camera_center] mass:[0.00]kg
 [13/24] [tcp_link] mass:[0.32]kg
 [14/24] [rh_p12_rn_r1] mass:[0.07]kg
 [15/24] [rh_p12_rn_r2] mass:[0.02]kg
 [16/24] [rh_p12_rn_l1] mass:[0.07]kg
 [17/24] [rh_p12_rn_l2] mass:[0.02]kg
 [18/24] [body_tray_src] mass:[4.34]kg
 [19/24] [body_tray_dst] mass:[4.34]kg
 [20/24] [body_obj_wrench] mass:[0.09]kg
 [21/24] [body_obj_screwdriver] mass:[0.07]kg
 [22/24] [body_

### Replay (close the viewer window to stop)

In [12]:
step = 0
iter_dataloader = iter(dataloader)
PnPEnv.reset()

while PnPEnv.env.is_viewer_alive():
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        data = next(iter_dataloader)
        if step == 0:
            PnPEnv.set_instruction(data['task'][0])
            PnPEnv.set_obj_pose(data['obj_init'][0].numpy())      # 16 values: 4 tools x (x,y,z,yaw)
        action = data['action'].numpy()                            # (1, 8): 6 joints, slide, gripper
        PnPEnv.step(action[0])

        rgb_agent = (data['observation.image'][0].numpy() * 255).astype(np.uint8)
        rgb_ego = (data['observation.wrist_image'][0].numpy() * 255).astype(np.uint8)
        PnPEnv.rgb_agent = np.transpose(rgb_agent, (1, 2, 0))      # 3,256,256 -> 256,256,3
        PnPEnv.rgb_ego = np.transpose(rgb_ego, (1, 2, 0))
        PnPEnv.rgb_side = np.zeros((480, 640, 3), dtype=np.uint8)
        PnPEnv.render()
        step += 1

        if step == len(episode_sampler):                           # loop the episode
            iter_dataloader = iter(dataloader)
            PnPEnv.reset()
            step = 0

DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION
DONE INITIALIZATION


In [6]:
PnPEnv.env.close_viewer()

### [Optional] write stats.json (needed by some lerobot versions)

In [7]:
write_json(serialize_dict(dataset.meta.stats), dataset.root / 'meta' / 'stats.json')